# CV Review Agent
## ATS Compliance Coach

This agent analyzes your CV/resume and provides actionable feedback using an Agent Loop with Tool Calls.

What it does:
- ATS Compliance Checks — section headers, contact info, formatting, parsability
- Keyword Gap Analysis — compares your CV to job description keywords
- Issue Tracking — records every problem found with severity and fix suggestion
- Report Generation — saves a comprehensive review report to a file

In [ ]:
# Imports and setup

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import re
import requests
from pypdf import PdfReader

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Pushover notifications

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    """Send a push notification if Pushover is configured."""
    print(f"Push: {message}")
    if pushover_user and pushover_token:
        requests.post(pushover_url, data={"user": pushover_user, "token": pushover_token, "message": message})

In [ ]:
# Global state for tracking issues across the review
issues = []

def extract_cv_text(file_path):
    """Read CV from PDF or text file and return its content."""
    try:
        if file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            text = ""
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t
        else:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        return {"success": True, "text": text, "word_count": len(text.split())}
    except FileNotFoundError:
        return {"success": False, "error": f"File not found: {file_path}"}

In [ ]:
def check_ats_formatting(cv_text):
    """Run ATS compliance checks: sections, length, contact info, action verbs, metrics."""
    results = {}

    # Check for standard section headers
    standard_sections = ["summary", "experience", "education", "skills", "contact"]
    results["found_sections"] = [s for s in standard_sections if s in cv_text.lower()]
    results["missing_sections"] = [s for s in standard_sections if s not in cv_text.lower()]

    # Word count (ideal: 400-800 for 1-2 pages)
    word_count = len(cv_text.split())
    results["word_count"] = word_count
    results["length_verdict"] = "too short" if word_count < 300 else "too long" if word_count > 1000 else "good"

    # Contact info
    results["has_email"] = bool(re.search(r'\b[\w.-]+@[\w.-]+\.\w+\b', cv_text))
    results["has_phone"] = bool(re.search(r'[\+\(]?[0-9][\d \-\(\)]{7,}\d', cv_text))

    # Action verbs
    action_verbs = ["led", "managed", "developed", "designed", "implemented", "achieved",
                    "increased", "reduced", "created", "launched", "built", "delivered",
                    "optimized", "streamlined", "mentored", "spearheaded", "orchestrated"]
    found_verbs = [v for v in action_verbs if v in cv_text.lower()]
    results["action_verbs_found"] = found_verbs
    results["action_verb_count"] = len(found_verbs)

    # Quantified achievements
    results["has_quantified_achievements"] = bool(re.search(r'\d+%|\$[\d,]+|\d+\+', cv_text))

    return results

In [ ]:
def check_keyword_match(cv_text, job_keywords):
    """Check how well CV keywords match job requirement keywords."""
    cv_lower = cv_text.lower()
    keywords = [kw.strip().lower() for kw in job_keywords.split(",") if kw.strip()]
    matched = [kw for kw in keywords if kw in cv_lower]
    missing = [kw for kw in keywords if kw not in cv_lower]
    score = round(len(matched) / len(keywords) * 100, 1) if keywords else 0
    return {"matched_keywords": matched, "missing_keywords": missing, "match_score_pct": score}


def record_issue(category, severity, description, suggestion):
    """Record an issue found during the CV review."""
    issue = {"category": category, "severity": severity, "description": description, "suggestion": suggestion}
    issues.append(issue)
    return {"recorded": "ok", "total_issues": len(issues)}


def save_report(filename, report_content):
    """Save the final CV review report to a file."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report_content)
    push(f"CV review saved to {filename} — {len(issues)} issues found")
    return {"saved": filename, "total_issues": len(issues)}

In [ ]:
# Tool JSON schemas — these tell the LLM what tools exist and what arguments they take

extract_cv_text_json = {
    "name": "extract_cv_text",
    "description": "Read a CV/resume from a PDF or text file and return its text content",
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {"type": "string", "description": "Path to the CV file (PDF or .txt)"}
        },
        "required": ["file_path"],
        "additionalProperties": False
    }
}

check_ats_formatting_json = {
    "name": "check_ats_formatting",
    "description": "Run ATS compliance checks on CV text: section headers, length, contact info, action verbs, quantified achievements",
    "parameters": {
        "type": "object",
        "properties": {
            "cv_text": {"type": "string", "description": "The full text content of the CV to analyze"}
        },
        "required": ["cv_text"],
        "additionalProperties": False
    }
}

check_keyword_match_json = {
    "name": "check_keyword_match",
    "description": "Compare CV text against comma-separated job requirement keywords and return a match score",
    "parameters": {
        "type": "object",
        "properties": {
            "cv_text": {"type": "string", "description": "The full text of the CV"},
            "job_keywords": {"type": "string", "description": "Comma-separated keywords from the job description"}
        },
        "required": ["cv_text", "job_keywords"],
        "additionalProperties": False
    }
}

record_issue_json = {
    "name": "record_issue",
    "description": "Record a specific issue found in the CV during review",
    "parameters": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Issue category: formatting, content, keywords, impact, or structure"},
            "severity": {"type": "string", "description": "Issue severity: critical, major, or minor"},
            "description": {"type": "string", "description": "What the issue is"},
            "suggestion": {"type": "string", "description": "How to fix or improve this issue"}
        },
        "required": ["category", "severity", "description", "suggestion"],
        "additionalProperties": False
    }
}

save_report_json = {
    "name": "save_report",
    "description": "Save the final CV review report to a file",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {"type": "string", "description": "Output filename for the report"},
            "report_content": {"type": "string", "description": "The full text content of the review report"}
        },
        "required": ["filename", "report_content"],
        "additionalProperties": False
    }
}

In [ ]:
# Tools list + handler + agent loop (same pattern as Lab 4 and Extra)

tools = [
    {"type": "function", "function": extract_cv_text_json},
    {"type": "function", "function": check_ats_formatting_json},
    {"type": "function", "function": check_keyword_match_json},
    {"type": "function", "function": record_issue_json},
    {"type": "function", "function": save_report_json},
]


def handle_tool_calls(tool_calls):
    """Dispatch tool calls using globals()"""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"error": f"Unknown tool: {tool_name}"}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results


def loop(messages):
    """The Agent Loop"""
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            results = handle_tool_calls(message.tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
system_prompt = """You are an expert CV/Resume reviewer and ATS (Applicant Tracking System) compliance coach.
Your job is to thoroughly analyze a candidate's CV and provide critical, actionable feedback.

Your Review Process:
1. First, use extract_cv_text to read the CV file
2. Run check_ats_formatting to get programmatic ATS compliance data
3. If job keywords are provided, run check_keyword_match to assess keyword fit
4. Analyze the CV content yourself for: weak bullet points, vague language, missing impact/metrics, poor structure
5. Use record_issue for EVERY problem you find — be thorough, find at least 5 issues
6. Save a comprehensive report using save_report

Evaluation Criteria:
- ATS Compliance: proper section headers, parseable format, contact info present
- Impact & Metrics: achievements quantified with numbers, percentages, dollar amounts
- Action Verbs: each bullet starts with a strong action verb (led, built, increased, etc.)
- Relevance: content matches the target role (if job keywords provided)
- Brevity & Clarity: no filler words, each bullet is concise and meaningful
- Structure: reverse chronological, consistent formatting, appropriate length (1-2 pages)

Be critical but constructive. After using your tools, provide a final summary with:
- Overall ATS readiness score (out of 100)
- Top 3 most critical improvements needed
- Specific rewrite suggestions for the weakest bullet points
"""

## Run the Agent

In [ ]:
issues = []


cv_file = "Emmanuel_Samuel_CV_AI_DS_ML.pdf" #replace with your CV file path

#comma-separated keywords from a job posting you're targeting
job_keywords = "python, machine learning, leadership, agile, AWS, data pipelines"

user_message = f"Please review the CV at '{cv_file}'."
if job_keywords.strip():
    user_message += f" Match it against these job keywords: {job_keywords}"

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}]
result = loop(messages)
print(result)

## Gradio Interface

Upload a CV and optionally paste job keywords to get an interactive review.

In [ ]:
import gradio as gr

def review_cv(cv_file, job_kw):
    global issues
    issues = []

    user_msg = f"Please review the CV at '{cv_file.name}'."
    if job_kw and job_kw.strip():
        user_msg += f" Match it against these job keywords: {job_kw}"

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_msg}]
    return loop(messages)

with gr.Blocks(title="CV Review Agent") as demo:
    gr.Markdown("CV Review Agent\nUpload your CV and get ATS-compliant, critical feedback.")
    with gr.Row():
        cv_input = gr.File(label="Upload CV (PDF or TXT)", file_types=[".pdf", ".txt"])
        kw_input = gr.Textbox(label="Job Keywords (optional, comma-separated)",
                              placeholder="python, leadership, agile, AWS, data pipelines...")
    output = gr.Textbox(label="Review Results", lines=25)
    btn = gr.Button("Review My CV", variant="primary")
    btn.click(fn=review_cv, inputs=[cv_input, kw_input], outputs=output)

demo.launch()